# 03 - Train DiT / Action Head From Scratch

Notebook này train một mini DiT/action head từ đầu trên dữ liệu đã chuẩn bị bởi notebook 02.

Mục tiêu CK:

- Load `states_train.npy` và `actions_train.npy` từ output notebook 02.
- Khởi tạo action module từ scratch.
- Train ít nhất 1 epoch bằng flow-matching loss.
- Lưu checkpoint, config, train log, loss curve, summary để dùng cho notebook 04/05 và báo cáo.

Notebook này cần GPU trên Kaggle.


In [ ]:
import csv
import json
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
SEED = 42

# Pipeline chính: full train set, 1 epoch. Nếu cần debug nhanh, tạm set MAX_TRAIN_SAMPLES hoặc MAX_STEPS_PER_EPOCH.
EPOCHS = 1
BATCH_SIZE = 1024
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 256
NUM_LAYERS = 4
NUM_HEADS = 4
DROPOUT = 0.1
GRAD_CLIP_NORM = 1.0
USE_AMP = True
NUM_WORKERS = 2
LOG_EVERY = 100

# None = dùng toàn bộ train samples từ notebook 02.
MAX_TRAIN_SAMPLES = None

# None = một epoch thật theo DataLoader. Đặt số nhỏ, ví dụ 500, chỉ khi cần chạy debug.
MAX_STEPS_PER_EPOCH = None

STAT_CHUNK_SIZE = 262_144
RUN_ROOT = Path('/kaggle/working/gr00t_dit_runs')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('run output:', RUN_ROOT)


In [ ]:
def find_prepared_root() -> Path:
    candidates = []
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for path in base.rglob('gr00t_prepared_3subsets'):
            if not path.is_dir():
                continue
            required = ['prepare_report.json', 'actions_train.npy', 'states_train.npy', 'train_samples.parquet']
            if all((path / name).exists() for name in required):
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            'Không tìm thấy gr00t_prepared_3subsets. Hãy Add Input output mới của notebook 02 trước khi chạy.'
        )
    # Ưu tiên Kaggle input đã Save Version, sau đó mới tới working.
    return sorted(candidates, key=lambda p: (0 if str(p).startswith('/kaggle/input') else 1, str(p)))[0]


PREPARED_ROOT = find_prepared_root()
print('PREPARED_ROOT:', PREPARED_ROOT)

with open(PREPARED_ROOT / 'prepare_report.json', 'r', encoding='utf-8') as f:
    prepare_report = json.load(f)

print(json.dumps({
    'subsets_found': prepare_report.get('subsets_found'),
    'max_frames_per_episode': prepare_report.get('max_frames_per_episode'),
    'action_cols': prepare_report.get('action_cols'),
    'state_cols': prepare_report.get('state_cols'),
    'num_train_samples': prepare_report.get('num_train_samples'),
    'num_test_samples': prepare_report.get('num_test_samples'),
    'action_dim': prepare_report.get('action_dim'),
    'state_dim': prepare_report.get('state_dim'),
}, ensure_ascii=False, indent=2))

assert prepare_report.get('action_cols') == ['action'], 'Output notebook 02 cũ: action_cols phải là ["action"].'
assert prepare_report.get('state_cols') == ['observation.state'], 'Output notebook 02 cũ: state_cols phải là ["observation.state"].'
assert prepare_report.get('action_dim') == prepare_report.get('state_dim'), 'Action/state dim lệch, hãy kiểm tra notebook 02.'
assert prepare_report.get('num_train_samples', 0) > 0, 'Không có train samples.'
assert prepare_report.get('num_test_samples', 0) > 0, 'Không có test samples.'


In [ ]:
actions_train = np.load(PREPARED_ROOT / 'actions_train.npy', mmap_mode='r')
states_train = np.load(PREPARED_ROOT / 'states_train.npy', mmap_mode='r')
train_samples = pd.read_parquet(PREPARED_ROOT / 'train_samples.parquet')

assert len(actions_train) == len(states_train) == len(train_samples)
action_dim = int(actions_train.shape[1])
state_dim = int(states_train.shape[1])

print('actions_train:', actions_train.shape, actions_train.dtype)
print('states_train:', states_train.shape, states_train.dtype)
print('train_samples:', train_samples.shape)
print(train_samples['__subset'].value_counts() if '__subset' in train_samples.columns else 'No subset column')


def compute_mean_std(arr: np.ndarray, chunk_size: int = STAT_CHUNK_SIZE):
    total = np.zeros(arr.shape[1], dtype=np.float64)
    total_sq = np.zeros(arr.shape[1], dtype=np.float64)
    n = 0
    for start in tqdm(range(0, len(arr), chunk_size), desc='Computing stats'):
        chunk = np.asarray(arr[start:start + chunk_size], dtype=np.float32)
        total += chunk.sum(axis=0, dtype=np.float64)
        total_sq += np.square(chunk, dtype=np.float64).sum(axis=0, dtype=np.float64)
        n += len(chunk)
    mean = total / max(n, 1)
    var = np.maximum(total_sq / max(n, 1) - mean ** 2, 1e-8)
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


state_mean, state_std = compute_mean_std(states_train)
action_mean, action_std = compute_mean_std(actions_train)

print('state mean/std:', state_mean.shape, state_std.min(), state_std.max())
print('action mean/std:', action_mean.shape, action_std.min(), action_std.max())


In [ ]:
class StateActionDataset(Dataset):
    def __init__(self, states, actions, state_mean, state_std, action_mean, action_std):
        self.states = states
        self.actions = actions
        self.state_mean = state_mean
        self.state_std = state_std
        self.action_mean = action_mean
        self.action_std = action_std

    def __len__(self):
        return len(self.actions)

    def __getitem__(self, idx):
        state = np.asarray(self.states[idx], dtype=np.float32)
        action = np.asarray(self.actions[idx], dtype=np.float32)
        state = (state - self.state_mean) / self.state_std
        action = (action - self.action_mean) / self.action_std
        return torch.from_numpy(state), torch.from_numpy(action)


base_dataset = StateActionDataset(states_train, actions_train, state_mean, state_std, action_mean, action_std)

if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(base_dataset):
    rng = np.random.default_rng(SEED)
    indices = rng.choice(len(base_dataset), size=int(MAX_TRAIN_SAMPLES), replace=False)
    train_dataset = Subset(base_dataset, indices.tolist())
else:
    train_dataset = base_dataset

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

print('Train dataset samples:', len(train_dataset))
print('Batches per epoch:', len(train_loader))


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device, dtype=t.dtype) / max(half - 1, 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class MiniDiTActionHead(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int, num_layers: int, num_heads: int, dropout: float):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.action_proj = nn.Linear(action_dim, hidden_dim)
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)

    def forward(self, noisy_action: torch.Tensor, state: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        time = self.time_embed(tau)
        action_token = self.action_proj(noisy_action) + time
        state_token = self.state_proj(state) + time
        tokens = torch.stack([action_token, state_token], dim=1)
        tokens = self.blocks(tokens)
        return self.out(self.norm(tokens[:, 0]))


model = MiniDiTActionHead(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print('params:', num_params, 'trainable:', trainable_params)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

config = {
    'prepared_root': str(PREPARED_ROOT),
    'output_root': str(RUN_ROOT),
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'num_heads': NUM_HEADS,
    'dropout': DROPOUT,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'use_amp': USE_AMP,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_steps_per_epoch': MAX_STEPS_PER_EPOCH,
    'state_dim': state_dim,
    'action_dim': action_dim,
    'num_params': int(num_params),
    'trainable_params': int(trainable_params),
    'train_dataset_samples': int(len(train_dataset)),
    'batches_per_epoch': int(len(train_loader)),
    'prepare_report': prepare_report,
}

with open(RUN_ROOT / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(json.dumps({k: config[k] for k in [
    'epochs', 'batch_size', 'learning_rate', 'hidden_dim', 'num_layers', 'num_heads',
    'state_dim', 'action_dim', 'num_params', 'train_dataset_samples', 'batches_per_epoch'
]}, indent=2))


In [ ]:
log_rows = []
global_step = 0
start_time = time.time()
model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_losses = []
    max_steps = len(train_loader) if MAX_STEPS_PER_EPOCH is None else min(int(MAX_STEPS_PER_EPOCH), len(train_loader))
    pbar = tqdm(train_loader, total=max_steps, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, (state, action) in enumerate(pbar, start=1):
        if step > max_steps:
            break
        state = state.to(device, non_blocking=True)
        action = action.to(device, non_blocking=True)

        tau = torch.rand(action.size(0), device=device)
        tau_view = tau[:, None]
        noise = torch.randn_like(action)
        noisy_action = tau_view * action + (1.0 - tau_view) * noise
        target_vector_field = noise - action

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
            pred = model(noisy_action, state, tau)
            loss = F.mse_loss(pred, target_vector_field)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if step == 1 or step % LOG_EVERY == 0 or step == max_steps:
            row = {
                'epoch': epoch,
                'step': step,
                'global_step': global_step,
                'loss': loss_value,
                'grad_norm': float(grad_norm.detach().cpu()),
                'lr': optimizer.param_groups[0]['lr'],
                'elapsed_sec': round(time.time() - start_time, 2),
            }
            log_rows.append(row)
            pbar.set_postfix(loss=f'{loss_value:.5f}')

    print(f'Epoch {epoch} mean loss: {np.mean(epoch_losses):.6f}')

print('Training finished. Global steps:', global_step)


In [ ]:
train_log_path = RUN_ROOT / 'train_log.csv'
with open(train_log_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['epoch', 'step', 'global_step', 'loss', 'grad_norm', 'lr', 'elapsed_sec'])
    writer.writeheader()
    writer.writerows(log_rows)

losses = [row['loss'] for row in log_rows]
steps = [row['global_step'] for row in log_rows]
plt.figure(figsize=(8, 4.5))
plt.plot(steps, losses, marker='o', linewidth=1)
plt.xlabel('Global step')
plt.ylabel('Flow-matching train loss')
plt.title('Mini DiT/action head training loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_ROOT / 'loss_curve.png', dpi=160)
plt.show()

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': config,
    'normalization': {
        'state_mean': state_mean,
        'state_std': state_std,
        'action_mean': action_mean,
        'action_std': action_std,
    },
    'global_step': global_step,
}
torch.save(checkpoint, RUN_ROOT / 'checkpoint_last.pt')

summary = {
    'status': 'completed',
    'objective': 'Train mini DiT/action head from scratch for CK implementation evidence.',
    'prepared_root': str(PREPARED_ROOT),
    'output_root': str(RUN_ROOT),
    'epochs_requested': EPOCHS,
    'epochs_completed': EPOCHS,
    'global_steps': int(global_step),
    'full_epoch_completed': MAX_TRAIN_SAMPLES is None and MAX_STEPS_PER_EPOCH is None,
    'num_train_samples': int(len(train_dataset)),
    'batch_size': BATCH_SIZE,
    'state_dim': state_dim,
    'action_dim': action_dim,
    'num_params': int(num_params),
    'first_logged_loss': float(losses[0]) if losses else None,
    'last_logged_loss': float(losses[-1]) if losses else None,
    'elapsed_sec': round(time.time() - start_time, 2),
    'artifacts': [
        'config.json',
        'train_log.csv',
        'loss_curve.png',
        'checkpoint_last.pt',
        'train_summary.json',
    ],
}

with open(RUN_ROOT / 'train_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Saved artifacts:')
for path in sorted(RUN_ROOT.iterdir()):
    print('-', path.name, round(path.stat().st_size / (1024 ** 2), 3), 'MB')


## Sau Khi Chạy Xong

Kiểm tra tab Output có thư mục:

```text
gr00t_dit_runs/
  config.json
  train_log.csv
  loss_curve.png
  checkpoint_last.pt
  train_summary.json
```

Nếu `train_summary.json` có `full_epoch_completed: true`, đây là bằng chứng notebook 03 đã train đủ 1 epoch trên toàn bộ train split từ notebook 02.

Save Version output này rồi dùng làm input cho notebook 04 evaluation.
